In [2]:
import pandas as  pd
import matplotlib.pyplot as plt
movies_df = pd.read_csv('../data/movies.csv')
ratings_df = pd.read_csv('../data/ratings.csv')

In [3]:
movie_stats = ratings_df.groupby('movieId')['rating'].agg(['count', 'mean']).reset_index()
movie_stats.columns = ['movieId', 'rating_count', 'rating_mean']
movie_stats_filtered = movie_stats[movie_stats['rating_count'] >= 50]
print(movie_stats_filtered.head())


   movieId  rating_count  rating_mean
0        1           215     3.920930
1        2           110     3.431818
2        3            52     3.259615
5        6           102     3.946078
6        7            54     3.185185


In [4]:
movie_matrix = movie_stats_filtered.merge(ratings_df, on='movieId', how='left')
movie_matrix = movie_matrix.pivot_table(index='userId', columns='movieId', values='rating')
print(movie_matrix.shape)


(606, 450)


In [5]:
user_means = movie_matrix.mean(axis=1)

movie_matrix_centered = movie_matrix.sub(user_means, axis=0)
movie_matrix_norm = movie_matrix_centered.fillna(0)


In [6]:
from sklearn.metrics.pairwise import cosine_similarity
movie_similarity = cosine_similarity(movie_matrix_norm.T)
movie_similarity_df = pd.DataFrame(movie_similarity, index=movie_matrix_norm.columns, columns=movie_matrix_norm.columns)
print(movie_similarity_df.shape)
print(movie_similarity_df.head())

(450, 450)
movieId    1         2         3         6         7         10        11      \
movieId                                                                         
1        1.000000  0.017058  0.004384  0.015428 -0.005029 -0.123372  0.010850   
2        0.017058  1.000000  0.119510 -0.042561  0.054620 -0.103048  0.049671   
3        0.004384  0.119510  1.000000 -0.016012  0.076507 -0.095099 -0.032070   
6        0.015428 -0.042561 -0.016012  1.000000  0.032444  0.102379  0.054094   
7       -0.005029  0.054620  0.076507  0.032444  1.000000  0.048436  0.208089   

movieId    16        17        19      ...    91500     91529     96079   \
movieId                                ...                                 
1       -0.061576  0.020359 -0.049214  ... -0.075259 -0.008759 -0.072917   
2       -0.173475  0.039398  0.069857  ... -0.036646 -0.099166 -0.097507   
3       -0.099392 -0.043256  0.253406  ... -0.071853 -0.040153 -0.064862   
6        0.258498 -0.025243 -0.075104  ..

In [7]:
def recommend_collaborative(film_title, n=5):
    # 1. Trouver le movieId correspondant au titre
    movie_row = movies_df[movies_df['title'].str.contains(film_title, case=False, na=False)]
    
    if movie_row.empty:
        return f"Désolé, le film '{film_title}' n'a pas été trouvé."
    
    movie_id = movie_row.iloc[0]['movieId']

    # 2. Vérifier si le film est dans notre matrice de similarité 
    # (Rappel : on a filtré les films avec < 50 notes)
    if movie_id not in movie_similarity_df.columns:
        return f"Le film '{movie_row.iloc[0]['title']}' a trop peu de notes pour générer des recommandations fiables."

    # 3. Récupérer le vecteur de similarité et trier
    similar_scores = movie_similarity_df[movie_id].sort_values(ascending=False)

    # 4. Extraire le top N (en excluant le premier qui est le film lui-même)
    top_n_ids = similar_scores.iloc[1:n+1].index.tolist()
    top_n_scores = similar_scores.iloc[1:n+1].values

    # 5. Créer le résultat avec les titres
    recommendations = movies_df[movies_df['movieId'].isin(top_n_ids)].copy()
    
    # On ajoute les scores pour transparence
    # On mappe les scores aux IDs pour garder le bon ordre
    recommendations['similarity_score'] = recommendations['movieId'].map(lambda x: similar_scores[x])
    
    return recommendations[['title', 'similarity_score']].sort_values(by='similarity_score', ascending=False)

# --- TEST ---
print(recommend_collaborative("Toy Story", n=5))



                                            title  similarity_score
2355                           Toy Story 2 (1999)          0.292575
506                                Aladdin (1992)          0.261117
7355                           Toy Story 3 (2010)          0.246225
868   Wallace & Gromit: The Wrong Trousers (1993)          0.211116
5374                      Incredibles, The (2004)          0.197818


In [8]:
from surprise import SVD, Dataset, Reader, accuracy
from surprise.model_selection import train_test_split

# 1. Définir l'échelle des notes (MovieLens est sur 0.5 - 5.0)
reader = Reader(rating_scale=(0.5, 5.0))

# 2. Charger les données (On utilise les ratings filtrés ou complets)
# Note: Surprise gère très bien les matrices creuses, on peut utiliser tout le dataset
data = Dataset.load_from_df(ratings_df[['userId', 'movieId', 'rating']], reader)

# 3. Diviser en train/test (80% / 20%) pour la comparaison
trainset, testset = train_test_split(data, test_size=0.2)


In [9]:
# Initialisation du modèle SVD
svd_model = SVD(n_factors=100, n_epochs=20, lr_all=0.005, reg_all=0.02)

# Entraînement
svd_model.fit(trainset)

# Prédiction sur le jeu de test
predictions = svd_model.test(testset)

# Calcul de l'erreur RMSE (Root Mean Squared Error)
rmse_svd = accuracy.rmse(predictions)
print(f"RMSE du modèle SVD : {rmse_svd}")


RMSE: 0.8629
RMSE du modèle SVD : 0.8628565291051203
